# 25 — DeepAgent + live chat

`shipit_agent.deep.DeepAgent` is the power-user `create_deep_agent` factory. It is a strict superset of LangChain's `deepagents`:

- Same one-liner ergonomics (`create_deep_agent(...)`)
- 7 deep tools instead of LangChain's 3 (`plan_task`, `decompose_problem`, `workspace_files`, `sub_agent`, `synthesize_evidence`, `decision_matrix`, `verify_output`)
- Built-in `verify=True`, `reflect=True`, `goal=Goal(...)`, and `rag=` integrations
- A modern multi-agent interactive chat REPL via `shipit chat`
- A programmatic `agent.chat()` helper for live multi-turn chat from Python

This notebook tours all of it.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent.deep import DEEP_AGENT_PROMPT, DeepAgent, Goal, create_deep_agent
from shipit_agent.llms import SimpleEchoLLM
from shipit_agent.rag import RAG, HashingEmbedder
from IPython.display import display, Markdown

from examples.run_multi_tool_agent import build_llm_from_env
llm = build_llm_from_env('bedrock')

In [10]:
llm

## 1. The opinionated prompt

In [11]:
print(DEEP_AGENT_PROMPT[:1000], '...')

You are a Deep Agent — a careful, methodical assistant that succeeds at long, multi-step tasks by **planning, verifying, and managing context** with discipline. When the user has wired specialised sub-agents into you, you can delegate focused work to them via `delegate_to_agent` (see the tool's description for the list).

## Core habits

1. **Plan before acting.** For any non-trivial task, call `plan_task` first. Write the plan as ordered, verifiable steps.
2. **Decompose hard reasoning.** Use `decompose_thought` when a question is multi-part — break it into sub-claims, address each.
3. **Use the workspace as durable memory.** Call `workspace_files` to write notes, intermediate results, and artifacts. Read them back when context scrolls out.
4. **Delegate side-quests.** When a sub-task is well-scoped (summarisation, fact-check, focused research), call `sub_agent` for a one-shot inner agent. When there is a *named* specialist available (`delegate_to_agent` lists them), prefer that — it 

## 2. Build a DeepAgent — one line

Seven deep tools are wired automatically. `with_builtins()` adds the rest of the builtin catalogue (web search, code execution, integrations).

In [12]:
agent = DeepAgent(llm=llm)
deep_tool_names = sorted({t.name for t in agent.tools})
print(f'{len(deep_tool_names)} tools wired:')
for name in deep_tool_names:
    print(f'  · {name}')

7 tools wired:
  · decision_matrix
  · decompose_problem
  · plan_task
  · sub_agent
  · synthesize_evidence
  · verify_output
  · workspace_files


In [13]:
full = DeepAgent.with_builtins(llm=llm)
names = {t.name for t in full.tools}
print(f'{len(names)} tools when use_builtins=True')
deep_only = {'plan_task', 'decompose_problem', 'workspace_files', 'sub_agent', 'synthesize_evidence', 'decision_matrix', 'verify_output'}
print('all 7 deep tools present:', deep_only <= names)

26 tools when use_builtins=True
all 7 deep tools present: True


## 3. The functional `create_deep_agent` helper

Drop-in alternative for users coming from LangChain. Plain Python functions are auto-wrapped as tools.

In [14]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

weather_agent = create_deep_agent(
    tools=[get_weather],
    llm=llm,
    system_prompt='You are a helpful weather assistant.',
)
print('tools:', sorted(t.name for t in weather_agent.tools))
assert 'get_weather' in {t.name for t in weather_agent.tools}
print('weather_agent is a DeepAgent:', isinstance(weather_agent, DeepAgent))

tools: ['decision_matrix', 'decompose_problem', 'get_weather', 'plan_task', 'sub_agent', 'synthesize_evidence', 'verify_output', 'workspace_files']
weather_agent is a DeepAgent: True


## 4. Power flags — verify, reflect, RAG, goal

Each flag is a single keyword argument. They compose freely.

In [15]:
rag = RAG.default(embedder=HashingEmbedder(dimension=512))
rag.index_text('Shipit supports Python 3.10+.', source='readme.md')
rag.index_text('DeepAgent ships seven tools by default.', source='changelog.md')

powerful = DeepAgent.with_builtins(
    llm=llm,
    rag=rag,                    # auto-cited grounded answers
    verify=True,                # verifier runs after every answer
    reflect=False,              # set True to enable self-critique
    max_iterations=20,
    parallel_tool_execution=True,
    context_window_tokens=200_000,
)
print('verify =', powerful.verify)
print('rag attached =', powerful.rag is not None)
print('inner agent has rag_search =', any(t.name == 'rag_search' for t in powerful.agent.tools))
print('max_iterations =', powerful.agent.max_iterations)

verify = True
rag attached = True
inner agent has rag_search = True
max_iterations = 20


### 4.1 Verified run — verdict on `result.metadata['verification']`

In [16]:
result = powerful.run('What Python version does Shipit support?')
print('output:', result.output[:200])
print('verification verdict present:', 'verification' in result.metadata)
if 'verification' in result.metadata:
    print('verification text:', result.metadata['verification']['text'][:200])

output: You are a Deep Agent — a careful, methodical assistant that succeeds at long, multi-step tasks by **planning, verifying, and managing context** with discipline. When the user has wired specialised sub
verification verdict present: True
verification text: Verification failed:
- Answer is concrete and addresses the user's question.: missing


### 4.2 Goal-driven mode

Pass `goal=Goal(...)` and `DeepAgent.run()` delegates to a `GoalAgent` that decomposes the goal, runs the inner agent on each sub-task, and self-evaluates against the success criteria.

In [17]:
goal_agent = DeepAgent(
    llm=llm,
    goal=Goal(
        objective='Say hello and confirm Python version.',
        success_criteria=['greeting', 'mentions python'],
        max_steps=2,
    ),
    rag=rag
)
result = goal_agent.run('What Python version does Shipit support?')
print('goal_status =', result.goal_status)
print('criteria_met =', result.criteria_met)
print('steps_taken =', result.steps_taken)

goal_status = partial
criteria_met = [False, True]
steps_taken = 1


## 5. Live chat — `agent.chat()`

Every `DeepAgent` has a `.chat()` helper that returns an `AgentChatSession` for multi-turn conversation. Streaming events arrive token-by-token (or step-by-step, depending on the LLM).

In [18]:
chat = powerful.chat(session_id='demo-1')

# First turn — streaming
for event in chat.stream('What Python version do we support?'):
    if event.type in ('run_started', 'tool_called', 'tool_completed', 'rag_sources', 'run_completed'):
        print(f'  · [{event.type}] {event.message[:80]}')

# Second turn — blocking
result = chat.send('And what about deep agent features?')
print()
print('answer:', result.output[:200])
print()
print('history length:', len(chat.history()))

  · [run_started] Agent run started
  · [run_completed] Agent run completed

answer: You are a Deep Agent — a careful, methodical assistant that succeeds at long, multi-step tasks by **planning, verifying, and managing context** with discipline. When the user has wired specialised sub

history length: 6


### 5.1 Persistent chat across processes

Use `FileSessionStore` so conversations survive restarts:

```python
from shipit_agent.stores import FileSessionStore

agent = DeepAgent.with_builtins(
    llm=llm,
    session_store=FileSessionStore(root='~/.shipit/sessions'),
)
chat = agent.chat(session_id='user-42')
```

## 6. The `shipit chat` CLI

From a terminal:

```bash
shipit chat                                      # default: DeepAgent
shipit chat --agent agent                        # plain Agent
shipit chat --agent goal --goal 'Build a CLI'
shipit chat --rag-file docs/manual.pdf
shipit chat --provider anthropic --session-dir ~/.shipit/sessions
shipit chat --reflect --verify
```

Slash commands inside the REPL:

```
/help                show all slash commands
/agent <type>        switch agent type live
/agents              list available agent types
/tools               list the agent's tools
/sources             show RAG sources from the last turn
/index <path>        index a file into the active RAG
/rag                 show RAG stats
/goal <objective>    set a goal
/reflect on|off      toggle reflection (DeepAgent)
/verify on|off       toggle verification (DeepAgent)
/save <path>         save conversation
/load <path>         load conversation
/exit                quit
```

We can drive the same REPL programmatically — useful for tests and scripted demos:

In [19]:
from shipit_agent.chat_cli import ChatREPL

repl = ChatREPL(
    llm=llm,
    agent_type='deep',
    use_builtins=False,
    quiet=True,
)

scripted = iter(['/agents', '/tools', '/info', 'hello there', '/history', '/exit'])
log = []

def in_fn(_): return next(scripted)
def out(*a, **k): log.append(' '.join(str(x) for x in a))

exit_code = repl.run(input_fn=in_fn, output=out)
print('exit_code =', exit_code)
print()
print('--- captured log (first 25 lines) ---')
for line in log[:25]:
    print(line)

exit_code = 0

--- captured log (first 25 lines) ---
╭───────────────────────────────────╮
│ Shipit Agent — interactive chat   │
│ agent=deep  session=chat-d8753a6f │
╰───────────────────────────────────╯
type /help for commands  ·  /exit to quit

Available agent types:
  ○ agent           Plain Agent — direct tool use, fast
  ● deep            DeepAgent — planning, workspace, sub-agents, verification
  ○ goal            GoalAgent — decompose → execute → self-evaluate
  ○ reflective      ReflectiveAgent — generate → critique → revise
  ○ adaptive        AdaptiveAgent — can write new tools at runtime
  ○ supervisor      Supervisor — coordinates multiple worker agents
  ○ persistent      PersistentAgent — checkpointed long-running tasks

Switch with: /agent <type>
7 tools:
  · plan_task                    Generate a concrete execution plan with ordered steps, risks, and checkpoints.
  · decompose_problem            Break a problem into workstreams, assumptions, risks, evidence needs, and

## 7. Switching agent types live

The chat CLI lets a user flip between any agent type without restarting. Each switch rebuilds the agent in place.

In [20]:
from shipit_agent.chat_cli import AGENT_TYPES, ChatREPL, make_agent

print('All agent types you can switch to:')
for t in AGENT_TYPES:
    a = make_agent(t, llm=llm, use_builtins=False)
    print(f'  · {t:12s} → {a.__class__.__name__}')

All agent types you can switch to:
  · agent        → Agent
  · deep         → DeepAgent
  · goal         → GoalAgent
  · reflective   → ReflectiveAgent
  · adaptive     → AdaptiveAgent
  · supervisor   → Supervisor
  · persistent   → PersistentAgent


## 8. Why this is more powerful than LangChain `create_deep_agent`

| Capability | LangChain `deepagents` | Shipit `DeepAgent` |
| --- | --- | --- |
| Number of bundled deep tools | 3 | **7** |
| RAG with auto-cited sources | (BYO) | **`rag=`** |
| Verification loop | (BYO) | **`verify=True`** |
| Reflection / self-critique | (BYO) | **`reflect=True`** |
| Goal-driven mode | (BYO) | **`goal=Goal(...)`** |
| Decision matrix tool | (BYO) | bundled |
| Evidence synthesis tool | (BYO) | bundled |
| Multi-agent live chat REPL | (CLI add-on, single agent) | **`shipit chat` — every agent type** |
| Long-term memory | MemoryStore | `AgentMemory` (conversation+semantic+entity) |
| Parallel tool execution | (BYO) | first-class flag |
| Auto context compaction | LangGraph | `context_window_tokens` |
| Streaming events | ✅ | ✅ (15 event types) |

Same one-liner DX, vastly more behind it.